# Model Evaluation

**Name:** Pravesh Bahakari

This notebook compares different train-test split ratios and evaluates the kNN model using 5-fold cross-validation.

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

In [3]:
df = pd.read_csv("../data/breast_cancer.csv")

df = df.drop(columns=["id", "Unnamed: 32"], errors="ignore")

df["diagnosis"] = df["diagnosis"].map({
    "B": 0,
    "M": 1
})

X = df.drop("diagnosis", axis=1)
y = df["diagnosis"]

print("Dataset shape:", df.shape)
print("Features:", X.shape)
print("Target:", y.shape)

Dataset shape: (569, 31)
Features: (569, 30)
Target: (569,)


In [4]:
# Compare two train-test splits
for test_size in [0.2, 0.3]:

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=42
    )

    # Scale the data
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # Train the kNN model
    model = KNeighborsClassifier(n_neighbors=9)
    model.fit(X_train, y_train)

    # Check accuracy
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)

    print("Train-test split:", int((1-test_size)*100), "/", int(test_size*100))
    print("Accuracy:", round(accuracy * 100, 2), "%")
    print()

Train-test split: 80 / 20
Accuracy: 96.49 %

Train-test split: 70 / 30
Accuracy: 97.08 %



In [5]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline

# Create the kNN model
model = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=9)
)

# Apply 5-fold cross-validation
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(model, X, y, cv=kfold, scoring="accuracy")

# Display the results
for i, score in enumerate(scores):
    print("Fold", i + 1, "Accuracy:", round(score * 100, 2), "%")

print("\nAverage accuracy:", round(scores.mean() * 100, 2), "%")
print("Standard deviation:", round(scores.std() * 100, 2), "%")

Fold 1 Accuracy: 98.25 %
Fold 2 Accuracy: 94.74 %
Fold 3 Accuracy: 94.74 %
Fold 4 Accuracy: 98.25 %
Fold 5 Accuracy: 97.35 %

Average accuracy: 96.66 %
Standard deviation: 1.61 %


## Results and Discussion

The 70/30 split achieved 97.08% accuracy, while the 80/20 split achieved 96.49%. The difference was small, but the 70/30 split performed slightly better.

In 5-fold cross-validation, the average accuracy was 96.66%, with a standard deviation of 1.61%. The results were quite consistent across the five folds.

Cross-validation is useful because it tests the model on different parts of the dataset. It gives a more reliable idea of how the model may perform on unseen data.

Overall, the cross-validation result was close to both train-test split results, which suggests that our kNN model performs consistently on this dataset.